In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

In [2]:
path = DATA_CALLEJERO_FILE

colspecs = [
    (0, 2), (2, 5), (5, 7), (7, 10), (10, 11), (11, 13),
    (13, 20), (20, 25), (25, 30), (30, 42), (42, 47), (47, 48),
    (48, 52), (52, 53), (53, 57), (57, 58), (58, 59), (59, 61),
    (61, 69), (69, 70), (70, 72), (72, 75), (75, 76), (76, 78),
    (78, 85), (85, 110), (110, 135), (135, 160), (160, 165),
    (165, 190), (190, 195), (195, 245), (245, 257), (257, 262),
    (262, 263), (263, 267), (267, 268), (268, 272), (272, 273)
]

names = [
    "CPRO", "CMUM", "DIST", "SECC", "LSECC", "SUBSC",
    "CUN", "CVIA", "CPSVIA", "MANZ", "CPOS", "TINUM",
    "EIN", "CEIN", "ESN", "CESN", "TIPOINF", "CDEV",
    "FVAR", "CVAR", "DIST_V", "SECC_V", "LSECC_V", "SUBSC_V",
    "CUN_V", "NENTCOC", "NENTSIC", "NNCLEC", "CVIA_V", "NVIAC",
    "CPSVIA_V", "DPSVIA", "MANZ_V", "CPOS_V", "TINUM_V",
    "EIN_V", "CEIN_V", "ESN_V", "CESN_V"
]

# --- Lectura del fichero ---
df = pd.read_fwf(
    path,
    colspecs=colspecs,
    names=names,
    encoding="latin1",
    dtype=str
)

# --- Limpieza de espacios ---
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
df = df.replace(r"^\s*$", pd.NA, regex=True)

# --- Crear CUSEC (concatenando las 5 primeras columnas) ---
df["CUSEC"] = (
    df["CPRO"].fillna("") +
    df["CMUM"].fillna("") +
    df["DIST"].fillna("") +
    df["SECC"].fillna("") +
    df["LSECC"].fillna("")
).astype(str)

# --- Filtrar por provincias de Castilla y León ---
provincias_cyl = ["05", "09", "24", "34", "37", "40", "42", "47", "49"]
df = df[df["CPRO"].isin(provincias_cyl)]



C:\Users\Carlo\AppData\Local\Temp\ipykernel_19020\2664609731.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(r"^\s*$", pd.NA, regex=True)


In [3]:
# Del callejero unicamente me interesa el CUSEC, el CPOS, el NENTSIC y el NVIAC
df = df[["CUSEC", "CPOS", "NENTSIC", "NVIAC"]]

df = df.rename(columns={
    "CPOS": "Código postal",
    "NENTSIC": "Entidad",
    "NVIAC": "Calle"
})

df.sample(5)


,CUSEC,Código postal,Entidad,Calle
1194362,4022801001,40219,VILLAVERDE DE ISCAR,SECADERO
1407619,4704801001,47692,CEINOS DE CAMPOS,MERCADILLO
687532,2405801001,24225,SAN JUSTO DE LOS OTEROS,MAYOR
1451286,4925101001,49539,VILLALUBE,CRUZ
1446085,4915301001,49145,PERILLA DE CASTRO,CALLEJA


In [4]:
# Integro el ID_CUSEC a partir del GDF de secciones censales
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)
print(gdf_secciones.head())

df = df.merge(
    gdf_secciones[["CUSEC", "Seccion_id"]],
    on="CUSEC",
    how="left"
)

df["Seccion_id"] = df["Seccion_id"].astype("Int64")


        CUSEC                      NMUN   NPRO  Seccion_id  \
0  0500101001                   Adanero  Ávila           1   
1  0500201001                Adrada, La  Ávila           2   
2  0500201002                Adrada, La  Ávila           3   
3  0500501001                  Albornos  Ávila           4   
4  0500701001  Aldeanueva de Santa Cruz  Ávila           5   

                                            geometry  
0  POLYGON ((365705.918 4536187.034, 365958.915 4...  
1  POLYGON ((363065.743 4462346.46, 363062.106 44...  
2  POLYGON ((361529.181 4469725.932, 361631.182 4...  
3  POLYGON ((343504.663 4523882.125, 343549.661 4...  
4  POLYGON ((294940.799 4473589.074, 294982.799 4...  


In [5]:
# --- Guardar el resultado ---
os.makedirs(DATA_OUTPUTS_DIR, exist_ok=True)
ruta = os.path.join(DATA_OUTPUTS_DIR, "callejero_cyl.csv")

df.to_csv(
    ruta,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivo guardado en: {ruta}")
print(f"Total de registros: {len(df)}")
display(df.head())


✅ Archivo guardado en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\callejero_cyl.csv
Total de registros: 163716


,CUSEC,Código postal,Entidad,Calle,Seccion_id
0,0500101001,05296,ADANERO,REAL,1
1,0500101001,05296,ADANERO,REAL,1
2,0500101001,05296,ADANERO,HUERTOS (LOS),1
3,0500101001,05296,ADANERO,HUERTOS (LOS),1
4,0500101001,05296,ADANERO,LIBERTAD,1


In [6]:
path = DATA_CALLEJERO_FILE

colspecs = [
    (42, 47),(135, 160), (0, 2),
]

names = [
    "cod_postal", "des_localidad", "cod_provincia"
]

# --- Lectura del fichero ---
df = pd.read_fwf(
    path,
    colspecs=colspecs,
    names=names,
    encoding="latin1",
    dtype=str
)

# --- Limpieza de espacios ---
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
df = df.replace(r"^\s*$", pd.NA, regex=True)

ruta = os.path.join(DATA_OUTPUTS_DIR, "cps.csv")

df.to_csv(
    ruta,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

